# Script 1 — Raw-file inventory

Checks that every NHANES file the pipeline reads is present before anything
runs. File names are matched without regard to case (`DEMO_G.XPT` and
`demo_g.xpt` are the same file). A missing required file stops the notebook
with the list of what is absent.

- **Cycle folders** `data/raw/2011-2012/` (suffix `_G`) and
  `data/raw/2013-2014/` (suffix `_H`): the questionnaire, examination and
  laboratory files read by scripts 3 and 4_5.
- **`data/raw/`**: the two minute-level accelerometer files `PAXMIN_G` and
  `PAXMIN_H`, read by script 5.
- `GLU`, `FASTQX` and `ALB_CR` are carried into the master file by script
  4_5 but enter no analysis; they are reported as optional and do not stop
  the pipeline when absent.

Reads: `config.py` for the folder paths. Writes: nothing.

In [ ]:
from config import raw_root, raw_2011, raw_2013, raw_file

REQUIRED = {
    'script 3 (questionnaires, examination)': ['DEMO', 'CFQ', 'SLQ', 'DPQ', 'PAQ', 'PAXDAY', 'PAXHD', 'BMX'],
    'script 4_5 (laboratory)':               ['CBC', 'BIOPRO', 'GHB', 'TCHOL'],
}
OPTIONAL = {'script 4_5 (carried, not analysed)': ['GLU', 'FASTQX', 'ALB_CR']}

missing = []
for folder, suffix in ((raw_2011, 'G'), (raw_2013, 'H')):
    print(f'{folder.relative_to(raw_root.parent)}')
    for role, stems in list(REQUIRED.items()) + list(OPTIONAL.items()):
        for stem in stems:
            name = f'{stem}_{suffix}.XPT'
            try:
                raw_file(folder, name)
                print(f'  yes  {name:16s} {role}')
            except FileNotFoundError:
                optional = role in OPTIONAL
                print(f'  {"--" if optional else "NO":4s} {name:16s} {role}{" -- optional" if optional else ""}')
                if not optional:
                    missing.append(str(folder / name))

print(f'{raw_root.relative_to(raw_root.parent)}')
for stem in ('PAXMIN_G', 'PAXMIN_H'):
    name = f'{stem}.XPT'
    try:
        p = raw_file(raw_root, name)
        print(f'  yes  {name:16s} script 5 (minute-level accelerometry, {p.stat().st_size / 1e9:.1f} GB)')
    except FileNotFoundError:
        print(f'  NO   {name:16s} script 5 (minute-level accelerometry)')
        missing.append(str(raw_root / name))

if missing:
    raise FileNotFoundError('missing required NHANES files:\n  ' + '\n  '.join(missing))
print('\nevery required file is present')